In [33]:
from ..brief import *


ImportError: attempted relative import with no known parent package

In [ ]:
def image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        image_base64 = base64.b64encode(image_file.read()).decode("utf-8")
    return image_base64

In [27]:
chat_model = ChatOpenAI(
            api_key="sk-6f27c88843124ca5a0bfc4d6b500b011",
            base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
            model="qwen3.5-plus",
            temperature=0,  # Deterministic output for consistent behavior
            max_retries=3,  # Robust error handling for network issues
            extra_body = {
            "enable_thinking": True,
            "enable_search": True,
        }
        )

pic_path = './Rice_project/pics/fig2.png'
pic_64 = image_to_base64(pic_path)

message = HumanMessage(
    content = [
        {
            "type": "text",
            "text": "请描述这张图片"
        },
        {
            "type": "image_url",
            "image_url": {"url": f"data:image/png;base64,{pic_64}"}
        },
        
    ]
)

response = chat_model.invoke([message])



In [23]:
def read_code_file(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        code_content = file.read()
    return code_content

In [28]:
response

AIMessage(content='这是一张二维散点图，通常用于生物信息学分析，特别是单细胞RNA测序（scRNA-seq）的数据质量控制（QC）。以下是详细描述：\n\n**1. 坐标轴：**\n*   **横轴（X轴）：** 标签为 `total_counts`。刻度范围从 0 到 700,000，每 100,000 为一个标记单位。这通常代表每个样本（如细胞）中的总读数或总UMI（Unique Molecular Identifiers）数量。\n*   **纵轴（Y轴）：** 标签为 `n_genes_by_counts`。刻度范围从 0 到 25,000，每 5,000 为一个标记单位。这通常代表在每个样本中检测到的不同基因的数量。\n\n**2. 数据分布与趋势：**\n*   **点的形态：** 数据由无数个细小的灰色点组成。\n*   **整体趋势：** 数据点呈现出明显的正相关关系，形成一条从左下角向右上角延伸的曲线。\n*   **曲线特征：**\n    *   **密集区：** 在横轴 0 到 50,000 的区间内，数据点非常密集，形成一条深灰色的带状区域。这表明大部分样本的总读数较低。\n    *   **增长趋势：** 随着 `total_counts` 的增加，`n_genes_by_counts` 迅速上升。\n    *   **饱和趋势：** 当 `total_counts` 超过 100,000 后，曲线的斜率变缓，表明检测到的基因数量增长变慢，逐渐趋于饱和。最高点的基因数量大约在 22,000 到 23,000 左右。\n    *   **稀疏区：** 在横轴大于 300,000 的区域，数据点变得非常稀疏，只有零星的几个点，说明高测序深度的样本很少。\n\n**3. 总结：**\n这张图展示了测序深度（总计数）与检测到的基因数量之间的关系。它反映了随着测序量的增加，能检测到的基因种类也会增加，但最终会达到一个平台期。这种图常用于筛选低质量的细胞（读数太低）或异常细胞（如双细胞，读数异常高）。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1602, 'prompt_token

In [ ]:
read_code_file('../../BIA-Ghostcoder/ghostcoder/graph/executor.py')a

'import logging\nfrom ..utils import *\nfrom ..prompts import load_prompt_template\nfrom ..config import coder_config\n\nfrom typing import TypedDict, Annotated, Optional, Type, Any\n#langchain\nfrom langchain_core.language_models import LanguageModelLike\nfrom langchain_core.messages import HumanMessage, SystemMessage\nfrom langchain_core.output_parsers import JsonOutputParser\n#langgraph\nfrom langgraph.graph import StateGraph, START, END\nfrom langgraph.graph.state import CompiledStateGraph\nfrom langgraph.types import Checkpointer\nfrom langgraph.store.base import BaseStore\nfrom langgraph.checkpoint.memory import MemorySaver\nfrom langgraph.pregel import RetryPolicy\n#from langgraph.types import interrupt\n#Autogen executors\nfrom autogen_core import CancellationToken\nfrom autogen_core.code_executor import CodeBlock\nfrom autogen_ext.code_executors.docker import DockerCommandLineCodeExecutor\nfrom autogen_ext.code_executors.local import LocalCommandLineCodeExecutor\n\n#----------